# Fisher matrix without the LISA response

This example computes a full Fisher matrix for an EMRI without the LISA response, using the sky-averaged LISA sensitivity curve on the two polarisations $h_+$ and $h_\times$, and shows how SEF reports degenerate parameters. It covers:

1. Computing the Fisher matrix over 11 parameters, including the sky and spin angles.
2. The degeneracy report: without the response, the four angles cannot all be measured, and SEF says which combinations are unconstrained.
3. Removing the degeneracy by treating the angles as known, and computing the uncertainties of the other parameters.

The observation time is short (`T = 0.01` years) to keep the example fast. This notebook only needs FEW and StableEMRIFisher; `fisher_matrix_with_response.ipynb` computes the same Fisher matrix through the LISA response.

In [1]:
import numpy as np

import few
from few.waveform import FastSchwarzschildEccentricFlux

from stableemrifisher.fisher import StableEMRIFisher
from stableemrifisher.utils.analysis import check_fisher

use_gpu = False  # set to True to run on an NVIDIA GPU

if not use_gpu:
    # restrict FEW to its CPU backend
    cfg_set = few.get_config_setter(reset=True)
    cfg_set.enable_backends("cpu")

## Set up StableEMRIFisher

The waveform model is FEW's Schwarzschild eccentric flux model. With no `noise_model` or `ResponseWrapper`, SEF logs that it uses the sky-averaged sensitivity curve.

In [2]:
T = 0.01  # observation time, years
dt = 10.0  # sampling interval, seconds

sef = StableEMRIFisher(
    waveform_class=FastSchwarzschildEccentricFlux,
    waveform_class_kwargs=dict(
        inspiral_kwargs=dict(err=1e-11),  # trajectory integrator tolerance
        mode_selector_kwargs=dict(mode_selection_threshold=1e-5),
    ),
    T=T,
    dt=dt,
    deriv_type="stable",
    use_gpu=use_gpu,
)

No noise model or response wrapper provided.
Defaulting to the sky-averaged sensitivity curve


## Compute the full Fisher matrix

The spin `a` is left out because this Schwarzschild model has no spin, and `xI0` and `Phi_theta0` because the orbit is equatorial. All other parameters are included, also the sky angles (`qS`, `phiS`) and the spin orientation angles (`qK`, `phiK`). SEF searches a stable step size for each parameter from its default grid; `stats_for_nerds=True` prints the chosen steps.

In [3]:
wave_params = dict(
    m1=1e6,  # primary mass, solar masses
    m2=10.0,  # secondary mass, solar masses
    a=0.0,  # primary spin
    p0=9.5,  # initial semi-latus rectum
    e0=0.4,  # initial eccentricity
    xI0=1.0,  # cosine of the initial inclination
    dist=0.1,  # luminosity distance, Gpc
    qS=np.pi / 3,  # sky location, polar angle
    phiS=np.pi / 4,  # sky location, azimuthal angle
    qK=np.pi / 6,  # spin orientation, polar angle
    phiK=np.pi / 8,  # spin orientation, azimuthal angle
    Phi_phi0=1.0,  # initial azimuthal phase
    Phi_theta0=0.0,  # initial polar phase
    Phi_r0=0.0,  # initial radial phase
)

param_names = ["m1", "m2", "p0", "e0", "dist", "qS", "phiS", "qK", "phiK", "Phi_phi0", "Phi_r0"]

Fisher = sef(wave_params, param_names=param_names, der_order=4, Ndelta=8, stats_for_nerds=True)

Waveform Generated. SNR: 60.79354047081135
  m1         delta = 1.931e+01   relerr = 2.6e-12   (8 steps)             
  m2         delta = 1.931e-04   relerr = 2.4e-08   (8 steps)              
  p0         delta = 1.834e-04   relerr = 1.3e-08   (8 steps)              
  e0         delta = 7.723e-06   relerr = 2.4e-07   (8 steps)              
  dist       analytic derivative                                           
  qS         delta = 1.455e-04   relerr = 3.8e-13   (8 steps)              
  phiS       delta = 1.091e-04   relerr = 1.6e-14   (8 steps)                
  qK         delta = 7.275e-05   relerr = 8.0e-13   (8 steps)                
  phiK       delta = 2.826e-04   relerr = 2.6e-14   (8 steps)                
  Phi_phi0   analytic derivative                                             
  Phi_r0     analytic derivative                                             
Computing Fisher   : 100%|██████████| 11/11 [00:01<00:00,  8.23deriv/s]          
Degenerate parameters: a combi

## Degenerate parameters

Without the response, the waveform depends on the four angles `qS`, `phiS`, `qK`, `phiK` only through two combinations: the inclination of the orbit to the line of sight and the polarisation angle. Two combinations of the angles therefore have no effect on the waveform, and the Fisher matrix cannot be inverted.

SEF detects this from the eigenvalues of the Fisher matrix, after normalising it by its diagonal so that the result does not depend on the units of the parameters. It reports the parameters of each unconstrained combination (the warnings above) and leaves the Fisher matrix unchanged. The result is stored in `sef.fisher_check`; the threshold on the relative eigenvalue is `degeneracy_rtol` (default `1e-12`).

In [4]:
check = sef.fisher_check
print("positive definite:", check["positive_definite"])
for params, rel_eigenvalue in check["degenerate"]:
    print(f"unconstrained combination of {params}, relative eigenvalue {rel_eigenvalue:.1e}")

print(f"condition number of the Fisher matrix: {np.linalg.cond(Fisher):.1e}")

positive definite: False
unconstrained combination of ['qS', 'phiS', 'qK', 'phiK'], relative eigenvalue -1.4e-17
unconstrained combination of ['qS', 'phiS', 'qK', 'phiK'], relative eigenvalue 5.3e-17
condition number of the Fisher matrix: 5.5e+22


## Removing the degeneracy

If the angles are known, for example from an electromagnetic counterpart, the Fisher matrix of the remaining parameters is the full Fisher matrix without the rows and columns of the angles. This matrix is invertible, and its inverse gives the uncertainties of the remaining parameters.

To measure the angles themselves, include the LISA response, whose time-dependent orientation breaks the degeneracy; see `fisher_matrix_with_response.ipynb`.

In [5]:
angles = ["qS", "phiS", "qK", "phiK"]
kept = [name for name in param_names if name not in angles]
index = [param_names.index(name) for name in kept]

Fisher_kept = Fisher[np.ix_(index, index)]
print("positive definite:", check_fisher(Fisher_kept, kept)["positive_definite"])

sigma = np.sqrt(np.diag(np.linalg.inv(Fisher_kept)))
for name, s in zip(kept, sigma):
    print(f"{name:9s} = {wave_params[name]:.6g} +/- {s:.3g}")

positive definite: True
m1        = 1e+06 +/- 3.6e+03
m2        = 10 +/- 0.269
p0        = 9.5 +/- 0.00186
e0        = 0.4 +/- 0.0033
dist      = 0.1 +/- 0.00541
Phi_phi0  = 1 +/- 0.036
Phi_r0    = 0 +/- 0.0246


## What this shows

SEF reports degenerate parameters instead of silently returning a Fisher matrix that cannot be inverted, and names the parameters involved. Removing the degenerate parameters (or adding information that constrains them, such as the LISA response) gives an invertible Fisher matrix.